# Qwen-Based AI Framing

This notebook runs the Qwen sentence-level framing classifier for national AI strategy documents. File loading, caching, country selection, and output writing live in this notebook; `src/ai_policy/qwen_framing.py` only contains clean Qwen classification and scoring helpers.

## Setup

The working Qwen configuration follows the existing `../LLM` code: `DASHSCOPE_API_KEY` with the international DashScope OpenAI-compatible endpoint. `QWEN_API_KEY`, `QWEN_BASE_URL`, and `QWEN_MODEL` are also supported.

In [ ]:
from pathlib import Path
import csv
import os
import sys
import time

import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / 'src').exists():
    repo_root = repo_root.parent

src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

repo_root

In [ ]:
from ai_policy.semantic_framing import load_country_text, split_sentences, word_count
from ai_policy.qwen_framing import (
    DEFAULT_BASE_URL,
    DEFAULT_MODEL,
    LABELS,
    classify_sentence,
    summarize_labels,
)

api_key = os.getenv('QWEN_API_KEY') or os.getenv('DASHSCOPE_API_KEY')
base_url = os.getenv('QWEN_BASE_URL', DEFAULT_BASE_URL)
model = os.getenv('QWEN_MODEL', DEFAULT_MODEL)

{
    'api_key_is_set': bool(api_key),
    'base_url': base_url,
    'model': model,
    'labels': LABELS,
}

## Countries and Output Files

In [ ]:
G7_CHINA_COUNTRIES = [
    'United States',
    'China',
    'United Kingdom',
    'France',
    'Germany',
    'Italy',
    'Japan',
    'Canada',
]

outputs = repo_root / 'outputs'
outputs.mkdir(parents=True, exist_ok=True)

summary_path = outputs / 'qwen_ai_framing_summary.csv'
sentence_path = outputs / 'qwen_ai_framing_sentence_labels.csv'
g7_table_path = outputs / 'qwen_g7_china_framing_table.csv'

sentence_fields = ['Country', 'Sentence index', 'Label', 'Confidence', 'Rationale', 'Sentence']
summary_fields = [
    'Country', 'Sources', 'Word count', 'Sentence count',
    'AI-friendly sentence count', 'AI-cautious sentence count', 'Mixed sentence count',
    'Neutral sentence count', 'AI-friendly score', 'AI-cautious score',
    'AI-friendly per 1000 words', 'AI-cautious per 1000 words',
    'Net AI-friendly framing', 'Net AI-cautious framing', 'AI-friendly share',
]

G7_CHINA_COUNTRIES

## Notebook-Side Data Loading and Cache Helpers

These helpers keep corpus loading and CSV cache behavior visible in the notebook instead of hiding it in the source module.

In [ ]:
def load_cached_sentence_rows(path=sentence_path):
    if not path.exists():
        return {}
    with path.open(newline='', encoding='utf-8') as file:
        rows = list(csv.DictReader(file))
    return {(row['Country'], row['Sentence']): row for row in rows}


def append_sentence_row(row, path=sentence_path):
    exists = path.exists()
    with path.open('a', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=sentence_fields)
        if not exists:
            writer.writeheader()
        writer.writerow(row)


def write_summary(rows, path=summary_path):
    with path.open('w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=summary_fields)
        writer.writeheader()
        writer.writerows(rows)


def rebuild_summary_from_sentence_cache(sentence_cache):
    if sentence_cache.empty:
        return pd.DataFrame(columns=summary_fields)

    cached = {
        (row['Country'], row['Sentence']): row
        for row in sentence_cache.to_dict('records')
    }
    summary_rows = []
    for country in G7_CHINA_COUNTRIES:
        text, sources = load_country_text(repo_root, country)
        country_rows = []
        missing = []
        for index, sentence in enumerate(split_sentences(text), start=1):
            row = cached.get((country, sentence))
            if row:
                expanded_row = dict(row)
                expanded_row['Sentence index'] = index
                country_rows.append(expanded_row)
            else:
                missing.append(index)

        if not country_rows:
            continue
        if missing:
            print(f'{country}: {len(missing)} sentences missing from Qwen cache')

        summary_rows.append(
            summarize_labels(
                country_rows,
                country=country,
                sources=sources,
                word_count=word_count(text),
            )
        )

    summary_rows = sorted(summary_rows, key=lambda row: row['Net AI-friendly framing'], reverse=True)
    return pd.DataFrame(summary_rows, columns=summary_fields)


def load_outputs(rebuild_summary=True):
    sentences = pd.read_csv(sentence_path) if sentence_path.exists() else pd.DataFrame()
    if rebuild_summary and not sentences.empty:
        summary = rebuild_summary_from_sentence_cache(sentences)
        write_summary(summary.to_dict('records'))
    else:
        summary = pd.read_csv(summary_path) if summary_path.exists() else pd.DataFrame()
    return summary, sentences

## Run Qwen Framing

Start with the smoke test only if you want to verify the API. The smoke test classifies three Canada sentences and will produce a Canada-only summary. For G7 + China results, keep `RUN_SMOKE_TEST = False` and use the full-run cell below.

In [ ]:
def run_qwen_framing(
    countries,
    *,
    limit_sentences=None,
    sleep_seconds=0.0,
    use_cache=True,
    temperature=0.0,
    max_retries=3,
):
    if not api_key:
        raise RuntimeError('Set QWEN_API_KEY or DASHSCOPE_API_KEY before running Qwen framing.')

    cached = load_cached_sentence_rows() if use_cache else {}
    summary_rows = []

    for country in countries:
        text, sources = load_country_text(repo_root, country)
        country_sentences = split_sentences(text)
        if limit_sentences:
            country_sentences = country_sentences[:limit_sentences]

        country_rows = []
        for index, sentence in enumerate(country_sentences, start=1):
            cache_key = (country, sentence)
            cached_row = cached.get(cache_key)
            if cached_row:
                row = {
                    'Country': country,
                    'Sentence index': cached_row['Sentence index'],
                    'Label': cached_row['Label'],
                    'Confidence': cached_row['Confidence'],
                    'Rationale': cached_row['Rationale'],
                    'Sentence': sentence,
                }
                country_rows.append(row)
                continue

            result = classify_sentence(
                sentence,
                api_key=api_key,
                model=model,
                base_url=base_url,
                temperature=temperature,
                max_retries=max_retries,
            )
            row = {
                'Country': country,
                'Sentence index': index,
                'Label': result['label'],
                'Confidence': result['confidence'],
                'Rationale': result['rationale'],
                'Sentence': sentence,
            }
            append_sentence_row(row)
            cached[cache_key] = row
            country_rows.append(row)
            if sleep_seconds:
                time.sleep(sleep_seconds)

        summary_rows.append(
            summarize_labels(
                country_rows,
                country=country,
                sources=sources,
                word_count=word_count(text),
            )
        )
        print(f'Finished {country}: {len(country_rows)} sentences')

    summary_rows = sorted(summary_rows, key=lambda row: row['Net AI-friendly framing'], reverse=True)
    write_summary(summary_rows)
    return pd.DataFrame(summary_rows)

In [ ]:
RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    smoke_summary = run_qwen_framing(['Canada'], limit_sentences=3)
    display(smoke_summary)
else:
    print('Smoke test skipped.')

## G7 + China Full Run

Run this cell to classify the full eight-country corpus. The sentence-level CSV is a cache, so interrupted runs can resume without reclassifying sentences that are already saved.

In [ ]:
RUN_G7_CHINA = True

if RUN_G7_CHINA:
    summary = run_qwen_framing(G7_CHINA_COUNTRIES)
    display(summary)
else:
    print('Set RUN_G7_CHINA = True to classify the full G7 + China corpus with Qwen.')

## Load Outputs

In [ ]:
summary, sentences = load_outputs()
summary

In [ ]:
if not sentences.empty:
    display(sentences.head(10))
    display(sentences['Label'].value_counts(dropna=False).rename('count').to_frame())
else:
    print('No sentence-level Qwen output yet.')

## G7 + China Result Table

In [ ]:
if not summary.empty:
    g7_summary = summary[summary['Country'].isin(G7_CHINA_COUNTRIES)].copy()
    g7_summary = g7_summary.sort_values('Net AI-friendly framing', ascending=False)
    display(g7_summary[[
        'Country', 'Sentence count', 'AI-friendly sentence count', 'AI-cautious sentence count',
        'Mixed sentence count', 'Neutral sentence count', 'AI-friendly per 1000 words',
        'AI-cautious per 1000 words', 'Net AI-friendly framing', 'AI-friendly share'
    ]])
else:
    print('No Qwen summary output yet.')

In [ ]:
if not sentences.empty:
    cache_coverage = (
        sentences[sentences['Country'].isin(G7_CHINA_COUNTRIES)]
        .groupby(['Country', 'Label'])
        .size()
        .unstack(fill_value=0)
        .reindex(G7_CHINA_COUNTRIES)
        .fillna(0)
        .astype(int)
    )
    display(cache_coverage)
else:
    print('No sentence-level Qwen cache yet.')

## Visualize Country Scores

In [ ]:
if not summary.empty:
    plot_df = summary[summary['Country'].isin(G7_CHINA_COUNTRIES)].sort_values('Net AI-friendly framing')
    ax = plot_df.plot.barh(
        x='Country',
        y=['AI-friendly per 1000 words', 'AI-cautious per 1000 words'],
        figsize=(10, 5),
        color=['#2ca02c', '#d62728'],
    )
    ax.set_xlabel('Confidence-weighted score per 1,000 words')
    ax.set_ylabel('')
    ax.set_title('Qwen AI Framing: Friendly vs Cautious')
    ax.legend(loc='lower right')
    plt.tight_layout()

In [ ]:
if not summary.empty:
    plot_df = summary[summary['Country'].isin(G7_CHINA_COUNTRIES)].sort_values('Net AI-friendly framing')
    colors = ['#d62728' if x < 0 else '#1f77b4' for x in plot_df['Net AI-friendly framing']]
    ax = plot_df.plot.barh(
        x='Country',
        y='Net AI-friendly framing',
        figsize=(10, 5),
        color=colors,
        legend=False,
    )
    ax.axvline(0, color='#555', linewidth=1)
    ax.set_xlabel('AI-friendly minus AI-cautious per 1,000 words')
    ax.set_ylabel('')
    ax.set_title('Qwen Net AI-Friendly Framing')
    plt.tight_layout()

In [ ]:
label_order = ['AI-friendly', 'AI-cautious', 'mixed', 'neutral']

if not sentences.empty:
    comp = (
        sentences[sentences['Country'].isin(G7_CHINA_COUNTRIES)]
        .groupby(['Country', 'Label'])
        .size()
        .unstack(fill_value=0)
        .reindex(G7_CHINA_COUNTRIES)
        .reindex(columns=label_order, fill_value=0)
    )
    comp_share = comp.div(comp.sum(axis=1).replace(0, pd.NA), axis=0).fillna(0)
    ax = comp_share.plot(
        kind='barh',
        stacked=True,
        figsize=(10, 5),
        color={
            'AI-friendly': '#2ca02c',
            'AI-cautious': '#d62728',
            'mixed': '#ffbf00',
            'neutral': '#9aa0a6',
        },
    )
    ax.set_xlabel('Share of classified sentences')
    ax.set_ylabel('')
    ax.set_title('Qwen Sentence Label Composition by Country')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5))
    plt.tight_layout()

## Review Sentence Labels

In [ ]:
def review_examples(label=None, country=None, n=10):
    if sentences.empty:
        return pd.DataFrame()
    df = sentences.copy()
    if label:
        df = df[df['Label'] == label]
    if country:
        df = df[df['Country'] == country]
    cols = ['Country', 'Label', 'Confidence', 'Rationale', 'Sentence']
    return df.sort_values('Confidence', ascending=False)[cols].head(n)

review_examples(label='AI-cautious', n=10)

## Save Reporting Table

In [ ]:
if not summary.empty:
    report_cols = [
        'Country', 'AI-friendly per 1000 words', 'AI-cautious per 1000 words',
        'Net AI-friendly framing', 'AI-friendly share'
    ]
    g7_summary[report_cols].to_csv(g7_table_path, index=False)
    print(g7_table_path)

## Compare With Rule-Based Semantic Framing

In [ ]:
rule_path = outputs / 'semantic_ai_framing_summary.csv'
rule_summary = pd.read_csv(rule_path) if rule_path.exists() else pd.DataFrame()

if not summary.empty and not rule_summary.empty:
    compare = summary[['Country', 'Net AI-friendly framing']].merge(
        rule_summary[['Country', 'Net AI-friendly framing']],
        on='Country',
        suffixes=('_qwen', '_rules'),
    )
    compare['rank_qwen'] = compare['Net AI-friendly framing_qwen'].rank(ascending=False)
    compare['rank_rules'] = compare['Net AI-friendly framing_rules'].rank(ascending=False)
    display(compare.sort_values('rank_qwen'))
else:
    print('Run both Qwen framing and rule-based semantic framing to compare rankings.')